# Bioengineering Practicum · BIMB265125
## Meeting 2 — Basics of modelling biological systems with simple differential equations

**Universitas Gadjah Mada · Faculty of Biology · Master's Programme in Biology**  
Matin Nuhamunada, S.Si., M.Sc., Ph.D. · 18 September 2026

---

### Before you start

1. **File → Save a copy in Drive.** Work in *your* copy.
2. Fill in your name and group below.
3. When you are finished: **Runtime → Restart and run all**. If it does not run clean
   from top to bottom, it is not finished.
4. **File → Download → Download .ipynb**, then put it in
   `BIMB265125_<yourname>/M02_ode/notebooks/`.

### What you are building on

Last week you played the Rabbit & Fox game by hand. Three ideas from it come back today:

| In the game | Today |
|---|---|
| A fox had to land on a rabbit | The law of mass action: rate ∝ [A][B] |
| A sealed meadow could not recover | Closed vs open reaction networks |
| One turn = one generation | The solver's time step |

In [ ]:
NAME  = ''      # e.g. 'Siti Rahmawati'
GROUP = ''      # e.g. 'Group 3'

assert NAME and GROUP, 'Fill in NAME and GROUP before you go on.'
print(f'{NAME} — {GROUP}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('numpy', np.__version__)

---
## Task 1 — A closed reaction network

$$A \;\rightleftharpoons\; B \qquad v = k_1[A] - k_2[B]$$

$$\frac{d[A]}{dt} = -v \qquad\qquad \frac{d[B]}{dt} = +v$$

Nothing enters and nothing leaves, so $[A]+[B]$ must stay constant.
That is your correctness check — and it is free.

**Before you run it, write down your prediction:** what shape will $[A](t)$ have,
and what value will it settle at?

In [ ]:
k1, k2 = 1.2, 0.4          # parameters (per time unit)

def closed(t, y):
    """Return the DERIVATIVES, not the new values."""
    A, B = y
    v = k1*A - k2*B        # net rate of A -> B
    return [-v, +v]

y0 = [1.0, 0.0]            # initial condition: all A, no B

sol = solve_ivp(closed, (0, 10), y0, dense_output=True, rtol=1e-8, atol=1e-10)
t = np.linspace(0, 10, 400)
A, B = sol.sol(t)

fig, ax = plt.subplots()
ax.plot(t, A, lw=2.2, label='[A]')
ax.plot(t, B, lw=2.2, label='[B]')
ax.plot(t, A + B, '--', lw=2, label='[A] + [B]')
ax.set_xlabel('time'); ax.set_ylabel('concentration')
ax.set_title('Closed network: A = B')
ax.legend()
fig.savefig('M02_fig01_closed.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- the check ---------------------------------------------------------
drift = np.abs((A + B) - (y0[0] + y0[1])).max()
print(f'largest drift in total mass: {drift:.3e}')
assert drift < 1e-6, 'Mass is not conserved - look for a sign error in closed().'
print('PASS: mass is conserved')

# --- compare with the analytical equilibrium ---------------------------
total = y0[0] + y0[1]
A_eq = total * k2 / (k1 + k2)
B_eq = total * k1 / (k1 + k2)
print(f'predicted equilibrium:  [A] = {A_eq:.4f}   [B] = {B_eq:.4f}')
print(f'simulated at t = 10:    [A] = {A[-1]:.4f}   [B] = {B[-1]:.4f}')

**Q1.1** Did the simulation match the algebra? 
**Q1.2** Set `y0 = [0.5, 0.5]` and run again. The curves change — does the equilibrium? 
Explain why in one sentence.

*Your answer:*


---
## Task 2 — An open reaction network

$$\varnothing \;\xrightarrow{k_{in}}\; A \;\xrightarrow{k_a}\; B \;\xrightarrow{k_b}\; \varnothing$$

$$\frac{d[A]}{dt} = k_{in} - k_a[A] \qquad\qquad \frac{d[B]}{dt} = k_a[A] - k_b[B]$$

Same chemistry in the middle — but now there is a boundary that material crosses.

**Predict first:** will $[A]+[B]$ be flat this time? Why (not)?

In [ ]:
# NOTE: new names. Re-using k1 and k2 here would silently change the
# closed model above - a classic notebook bug. Keep parameter names distinct.
kin, ka, kb = 0.8, 1.2, 0.5

def open_sys(t, y):
    A, B = y
    return [kin - ka*A,         # dA/dt
            ka*A - kb*B]        # dB/dt

sol2 = solve_ivp(open_sys, (0, 10), [0.0, 0.0], dense_output=True, rtol=1e-8)
A2, B2 = sol2.sol(t)

fig, ax = plt.subplots()
ax.plot(t, A2, lw=2.2, label='[A]')
ax.plot(t, B2, lw=2.2, label='[B]')
ax.plot(t, A2 + B2, '--', lw=2, label='[A] + [B]')
ax.set_xlabel('time'); ax.set_ylabel('concentration')
ax.set_title('Open network: 0 -> A -> B -> 0')
ax.legend()
fig.savefig('M02_fig02_open.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'predicted steady state:  [A] = {kin/ka:.4f}   [B] = {kin/kb:.4f}')
print(f'simulated at t = 10:     [A] = {A2[-1]:.4f}   [B] = {B2[-1]:.4f}')

**Q2.1** The system settles down, but $[A]+[B]$ is *not* constant. 
What is constant instead, once it has settled? (Hint: look at the fluxes, not the amounts.) 

**Q2.2** This is the difference between an **equilibrium** and a **steady state**. 
Which one is a living cell in, and what does it cost the cell to stay there?

*Your answer:*


---
## Task 3 — What the solver is actually doing

`solve_ivp` did not solve the equation the way you would on paper. It took small steps.
Euler's method is the crudest version of that idea, and it is exactly what you did by hand
last week with $\Delta t$ = one generation:

$$y_{n+1} = y_n + \Delta t \cdot f(y_n)$$

In [ ]:
def euler(rhs, t_end, y0, dt):
    """Fixed-step Euler. The whole algorithm is the three lines in the loop."""
    y = np.array(y0, dtype=float)
    ts, ys = [0.0], [y.copy()]
    for i in range(int(t_end / dt)):
        dy = np.array(rhs(ts[-1], y))
        y = y + dt * dy
        ts.append((i + 1) * dt)
        ys.append(y.copy())
    return np.array(ts), np.array(ys)

fig, ax = plt.subplots()
ax.plot(t, A, 'k-', lw=2.6, label='solve_ivp (reference)')
for dt in (1.0, 0.5, 0.1):
    te, ye = euler(closed, 10, y0, dt)
    ax.plot(te, ye[:, 0], 'o-', ms=4, lw=1.4, alpha=0.85, label=f'Euler, dt = {dt}')
ax.set_xlabel('time'); ax.set_ylabel('[A]')
ax.set_title('Step size matters')
ax.legend()
fig.savefig('M02_fig03_euler.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# How wrong is each step size? Measure the LARGEST error along the whole
# trajectory - at t = 10 everything has already settled, so the end point
# flatters a bad solver.
prev = None
for dt in (1.0, 0.5, 0.25, 0.125, 0.0625):
    te, ye = euler(closed, 10, y0, dt)
    exact = sol.sol(te)[0]            # solve_ivp, evaluated at the same times
    err = np.abs(ye[:, 0] - exact).max()
    ratio = '' if prev is None else f'   (previous / this = {prev/err:.1f})'
    print(f'dt = {dt:<7} max error = {err:.3e}{ratio}')
    prev = err

**Q3.1** Halving `dt` does roughly what to the error? 
**Q3.2** Try `dt = 2.0`. Describe what happens, and explain it in terms of the step 
overshooting the equilibrium.

**Q3.3** Your game last week used $\Delta t$ = one generation. Given what you just saw, 
how much would you trust its numbers? How much would you trust its *shape*?

*Your answer:*


---
## Task 4 — Your game, as a differential equation

This is the open-ended task. Take the Lotka–Volterra model:

$$\frac{dR}{dt} = \alpha R - \beta RF \qquad\qquad \frac{dF}{dt} = \delta RF - \gamma F$$

* $R$ = rabbits (candies), $F$ = foxes (sticky notes)
* $\alpha$ = how fast rabbits breed · $\beta$ = how dangerous an encounter is
* $\delta$ = how efficiently a fox turns rabbits into foxes · $\gamma$ = how fast a fox starves

**Your job:** choose parameters so the curves resemble *your group's* Run 3 tally sheet,
then answer the questions underneath.

In [ ]:
def lotka_volterra(t, y, alpha, beta, delta, gamma):
    R, F = y
    dR = alpha*R - beta*R*F
    dF = delta*R*F - gamma*F
    return [dR, dF]

# ---- TUNE THESE to match your own tally sheet -------------------------
alpha, beta, delta, gamma = 0.55, 0.19, 0.043, 0.53
R0, F0 = 4.0, 1.0
# ----------------------------------------------------------------------

sol3 = solve_ivp(lotka_volterra, (0, 12), [R0, F0],
                 args=(alpha, beta, delta, gamma),
                 dense_output=True, rtol=1e-9)
tt = np.linspace(0, 12, 1000)
R, Fx = sol3.sol(tt)

# ---- PASTE YOUR OWN NUMBERS HERE from the Run 3 tally sheet -----------
gen        = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
rabbits    = [4, 8, 16, 26, 36, 36, 36, 36, 26, 18, 10, 12, 20]
foxes      = [1, 1, 1, 2, 2, 2, 4, 8, 10, 8, 4, 1, 1]
# ----------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(gen, rabbits, 'o', ms=8, label='game: rabbits')
ax.plot(gen, foxes, 's', ms=7, label='game: foxes')
ax.plot(tt, R, lw=2.2, alpha=0.6, label='ODE: R(t)')
ax.plot(tt, Fx, lw=2.2, alpha=0.6, label='ODE: F(t)')
ax.set_xlabel('generation / time'); ax.set_ylabel('population')
ax.set_title('The game and the equation, side by side')
ax.legend(ncol=2)
fig.savefig('M02_fig04_game_vs_ode.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# The phase portrait: F against R, with no time axis at all.
fig, ax = plt.subplots(figsize=(5, 4.5))
ax.plot(R, Fx, lw=2)
ax.plot(gamma/delta, alpha/beta, 'o', ms=10)
ax.annotate('coexistence steady state', (gamma/delta, alpha/beta),
            xytext=(8, 12), textcoords='offset points')
ax.set_xlabel('Rabbits R'); ax.set_ylabel('Foxes F')
ax.set_title('Phase portrait')
plt.show()

print(f'steady state: R* = {gamma/delta:.2f}, F* = {alpha/beta:.2f}')

### Write-up (about 150 words)

Answer all four:

1. Which parameter did you have to change most to match your tally sheet, and why?
2. **The sanctuary.** It is not in the equations above. Where would you put it — 
   in which term, and as what kind of change? (Hint: a fixed number of rabbits that 
   predation cannot reach.)
3. The ODE curves are smooth and repeatable; your tally sheet is lumpy and was different 
   for every group. Which of the two would you trust for a population of 6 rabbits, and 
   which for 10⁹ bacteria in a fermenter?
4. The grid held at most 36 candies. Which term would you add to the equation for $R$ to 
   represent that, and what is it called?

*Your answer:*


---
## Optional — add the carrying capacity yourself

If you have time, modify `lotka_volterra` so that the prey grows logistically:

$$\frac{dR}{dt} = \alpha R\left(1 - \frac{R}{K}\right) - \beta RF$$

with $K = 36$. Compare the phase portrait with the one above — the closed orbits
become a spiral. Why?

In [ ]:
# your code here


---
## Checklist before you submit

- [ ] `NAME` and `GROUP` are filled in
- [ ] **Runtime → Restart and run all** completes with no errors
- [ ] Four figures saved: `M02_fig01` … `M02_fig04`
- [ ] Every *Your answer* cell is filled in
- [ ] Downloaded as `.ipynb` into `M02_ode/notebooks/`

---

**Next week — Meeting 3:** Python data structures, visualisation and numerical
integration in more depth. We build directly on this notebook, so make sure it runs clean.